# R1-09 · Dashboards ligeros con Gradio — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea A · *Analista de Datos* · Semana 10 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Construir funciones de **filtro** y **KPIs** con pandas.
2. Crear una **visualización** reutilizable como componente.
3. Envolver todo en un **tablero Gradio** publicable desde Colab.
4. Juzgar **cuándo un dashboard aporta** y cuándo no.

**Competencia de salida:** publicar un tablero interactivo simple sobre datos públicos, sin instalar nada.

**Dato real:** compras públicas de alimentos por categoría y región (ChileCompra).

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

### Cargamos los datos (ya hecho arriba). Construyamos el tablero por partes.

## 1. Filtrar es la base de todo tablero

Un dashboard es, en el fondo, **filtrar** datos y mostrar el resultado. Empezamos por una función pura de pandas que filtra por categoría y región (`'(todas)'` = sin filtro).

### ✍️ Ejercicio 1 — Escribe la función `filtrar`

In [ ]:
def filtrar(df, categoria="(todas)", region="(todas)"):
    d = df.copy()
    # TODO: si categoria != "(todas)", filtra d por esa categoria
    # TODO: si region != "(todas)", filtra d por esa region
    return d

print("Sin filtro:", filtrar(df).shape[0])

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert filtrar(df).shape[0] == df.shape[0]
    sub = filtrar(df, categoria="Productos de panadería")
    assert set(sub["categoria"].unique()) == {"Productos de panadería"}
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "Compara d['categoria'] == categoria y d['region_comprador'] == region solo cuando no sea '(todas)'.")
    print("   Detalle:", e)

## 2. Los KPIs: los números grandes del tablero

Un buen tablero muestra **2-3 indicadores claros**. Calculemos gasto total, nº de órdenes y ticket promedio.

### ✍️ Ejercicio 2 — Escribe la función `kpis`

In [ ]:
def kpis(d):
    return {
        "gasto_total": ...,        # TODO: suma de monto_total
        "n_ordenes": ...,          # TODO: nº de filas
        "ticket_promedio": ...,    # TODO: promedio de monto_total
    }

print(kpis(filtrar(df)))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    k = kpis(df)
    assert k["n_ordenes"] == len(df)
    assert k["gasto_total"] > 0
    assert abs(k["ticket_promedio"] - df["monto_total"].mean()) < 1e-6
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "Usa d['monto_total'].sum(), len(d) y d['monto_total'].mean().")
    print("   Detalle:", e)

## 3. Una tabla resumen: gasto por región

El tercer ingrediente: agrupar y ordenar. Esta tabla alimentará el tablero.

### ✍️ Ejercicio 3 — Escribe `gasto_por_region`

In [ ]:
def gasto_por_region(d):
    # TODO: agrupa por region_comprador, suma monto_total y ordena de mayor a menor
    return ...

print(gasto_por_region(df).head(3))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    g = gasto_por_region(df)
    assert list(g.values) == sorted(g.values, reverse=True)
    assert len(g) == df["region_comprador"].nunique()
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "df.groupby('region_comprador')['monto_total'].sum().sort_values(ascending=False).")
    print("   Detalle:", e)

In [ ]:
# (ilustración) Gasto por región — así se vería el panel principal del tablero
g = gasto_por_region(df).head(10)
plt.figure(figsize=(6, 4))
plt.barh(g.index[::-1], g.values[::-1], color="#4240e5")
plt.xlabel("Gasto total (CLP)"); plt.title("Gasto por región (top 10)")
plt.tight_layout(); plt.show()

## 4. Un gráfico como componente del tablero

Gradio puede mostrar una figura de matplotlib. Escribe una función que **devuelva** la figura.

### ✍️ Ejercicio 4 — Escribe `figura_top_categorias`

In [ ]:
def figura_top_categorias(d, n=8):
    top = d.groupby("categoria")["monto_total"].sum().sort_values().tail(n)
    fig, ax = plt.subplots(figsize=(6, 4))
    # TODO: dibuja un barh con top.index y top.values y devuelve fig
    ...
    return fig

print(type(figura_top_categorias(df)).__name__)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    fig = figura_top_categorias(df)
    assert fig.__class__.__name__ == "Figure"
    assert len(fig.axes) >= 1
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "ax.barh(top.index, top.values) y recuerda 'return fig'.")
    print("   Detalle:", e)

## 5. Arma el tablero

In [ ]:
# ── El tablero con Gradio (se ejecuta en Colab) ───────────────────────────────
# gradio puede no estar instalado aquí; lo envolvemos para no romper la ejecución.
try:
    import gradio as gr

    def tablero(categoria, region):
        d = filtrar(df, categoria, region)
        k = kpis(d)
        resumen = f"Gasto total: ${k['gasto_total']:,.0f}  ·  Órdenes: {k['n_ordenes']:,}"
        return resumen, figura_top_categorias(d)

    cats = ["(todas)"] + sorted(df["categoria"].unique())
    regs = ["(todas)"] + sorted(df["region_comprador"].unique())
    demo = gr.Interface(
        tablero,
        [gr.Dropdown(cats, value="(todas)", label="Categoría"),
         gr.Dropdown(regs, value="(todas)", label="Región")],
        [gr.Text(label="Resumen"), gr.Plot(label="Top categorías")],
        title="Compras públicas — tablero de gasto",
    )
    # En Colab, descomenta para publicar un link público (sin túnel):
    # demo.launch(share=True)
    print("✔ App Gradio construida. En Colab ejecuta: demo.launch(share=True)")
except ModuleNotFoundError:
    print("ℹ Gradio no está instalado aquí. En Colab corre primero:  !pip -q install gradio")

## Cierre

- Un dashboard = **filtrar** + **KPIs** + **una visualización**, envueltos en una interfaz.
- Construye y prueba las **funciones de datos por separado** (como aquí) antes de armar la interfaz.
- **Gradio** corre dentro de Colab con `demo.launch(share=True)` y se publica gratis en Hugging Face Spaces.

> *¿Cuándo NO un dashboard? Si la pregunta se responde una sola vez, un gráfico o un informe basta. El tablero se justifica cuando alguien va a explorar y filtrar de forma recurrente.*